[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part2_rag/ch04_embeddings.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 4장 임베딩과 벡터 검색
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제2부 RAG — 검색으로 지식을 더하다
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
# 실습 라이브러리 설치 (검증 시점 2026-06, 하한 고정)
!pip install -q "google-genai>=1.0"

In [ ]:
# 설치된 핵심 라이브러리 버전 기록 (재현용) — 이 출력의 버전을 그대로 고정하면 가장 안정적이다
import importlib.metadata as _m
for _p in ["google-genai","langchain","langchain-google-genai","langchain-community",
           "langchain-text-splitters","langgraph","faiss-cpu","tiktoken","rank-bm25",
           "networkx","tavily-python"]:
    try: print(f"{_p}=={_m.version(_p)}")
    except Exception: pass

### API 키와 임베딩 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.5-flash"   # 모델 갱신 시 이 한 줄만 변경
EMBED_MODEL = "gemini-embedding-001"

## 예제 4-1. 문장을 임베딩 벡터로 바꾸기

In [ ]:
from google import genai

client = genai.Client(api_key=API_KEY)
res = client.models.embed_content(
    model=EMBED_MODEL,
    contents="휴가를 며칠 쓸 수 있나요",
)
vec = res.embeddings[0].values
print("차원:", len(vec))
print("앞 5개 값:", [round(x, 3) for x in vec[:5]])

## 예제 4-2. 코사인 유사도로 의미 비교

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def embed(text):
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text,
    )
    return res.embeddings[0].values

base = embed("휴가를 며칠 쓸 수 있나요")
for text in ["연차 한도는 며칠인가요", "오늘 환율이 얼마예요"]:
    print(f"{cosine(base, embed(text)):.3f}  {text}")

## 예제 4-3. 미니 의미 검색기

In [ ]:
docs = [
    "연차는 입사 1년 후 15일이 주어진다",
    "점심 시간은 12시부터 1시까지다",
    "재택근무는 주 2회까지 가능하다",
]
doc_vecs = [embed(d) for d in docs]

query = "휴가 며칠 쓸 수 있어"
q_vec = embed(query)
ranked = sorted(zip(docs, doc_vecs), key=lambda x: cosine(q_vec, x[1]), reverse=True)
print("질문:", query)
for doc, vec in ranked:
    print(f"{cosine(q_vec, vec):.3f}  {doc}")

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 4장 노트북